In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from moabb.paradigms import P300
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation

datasets = [
    BI2012(),
    BI2013a(),
    BI2014a(),
    BI2014b(),
    BI2015a(),
    BI2015b(),
    BNCI2014_008(),
    BNCI2014_009(),
    BNCI2015_003(),
    Cattan2019_VR(),
    EPFLP300(),
    ErpCore2021_LRP(),
    ErpCore2021_MMN(),
    ErpCore2021_N170(),
    ErpCore2021_N400(),
    ErpCore2021_P3(),
    Huebner2017(),
    Huebner2018(),
    Lee2019_ERP(),
]

paradigm = P300(
    resample=48,

)
cache_config = dict(
    use=True,
    save_raw=False,
    save_epochs=False,
    save_array=True,
    overwrite_raw=False,
    overwrite_epochs=False,
    overwrite_array=False,
)

In [3]:
import copy
from sklearn.base import clone
import dask
import os
from sklearn.preprocessing import FunctionTransformer
import tensorly as tl

def eval_moabb_within_session(dataset, subject, pipe):
    subj_dataset = copy.deepcopy(dataset)
    n_subjects = len(dataset.subject_list)
    subj_dataset.subject_list = [subject]
    evaluation = WithinSessionEvaluation(
        paradigm=paradigm,
        datasets=subj_dataset,
        overwrite=False,
        random_state=42,
        n_jobs=5,
        suffix=f'bttda_dask_dataset-{dataset.code}_subject-{subject}_pipe-{pipe}',
        cache_config=cache_config,
    )
    print(f'dataset={dataset.code}, subject={subject}/{n_subjects}, pipe={pipe}')
    return evaluation.process({pipe:clone(pipelines[pipe])})





In [4]:
import joblib
from joblib import Parallel, delayed
import distributed
from IPython import display
import pandas as pd
from classification_erp import get_pipelines
from hpc import create_cluster, create_client, TIMEOUT

pipelines = get_pipelines()

with create_cluster(cluster='cpu') as cluster, create_client(cluster) as client:
    results = []
    for dataset in datasets:
        print(f'Benchmarking on dataset {dataset.code}...')
        job_args = []
        for subject in dataset.subject_list:
            for pipe in pipelines.keys():
                job_args.append((dataset, subject,pipe))    
        with joblib.parallel_backend('dask', wait_for_workers_timeout=TIMEOUT): 
            results += Parallel(n_jobs=len(job_args), verbose=True)(delayed(eval_moabb_within_session)(*args) for args in job_args)
results = pd.concat(results, ignore_index=True)

Benchmarking on dataset BrainInvaders2012...


[Parallel(n_jobs=75)]: Using backend DaskDistributedBackend with 27 concurrent workers.
[Parallel(n_jobs=75)]: Done  75 out of  75 | elapsed:   42.0s finished
[Parallel(n_jobs=72)]: Using backend DaskDistributedBackend with 1341 concurrent workers.


Benchmarking on dataset BrainInvaders2013a...


[Parallel(n_jobs=72)]: Done  19 out of  72 | elapsed:    4.4s remaining:   12.3s
[Parallel(n_jobs=72)]: Done  72 out of  72 | elapsed:    8.3s finished
[Parallel(n_jobs=192)]: Using backend DaskDistributedBackend with 1350 concurrent workers.


Benchmarking on dataset BrainInvaders2014a...


...

[Parallel(n_jobs=39)]: Done  20 out of  39 | elapsed:    0.4s remaining:    0.4s
[Parallel(n_jobs=39)]: Done  39 out of  39 | elapsed:    0.5s finished
[Parallel(n_jobs=36)]: Using backend DaskDistributedBackend with 1350 concurrent workers.


Benchmarking on dataset Huebner2018...


[Parallel(n_jobs=36)]: Done  36 out of  36 | elapsed:    0.4s finished
[Parallel(n_jobs=162)]: Using backend DaskDistributedBackend with 1350 concurrent workers.


Benchmarking on dataset Lee2019-ERP...


[Parallel(n_jobs=162)]: Done  71 out of 162 | elapsed:    0.9s remaining:    1.1s
[Parallel(n_jobs=162)]: Done 162 out of 162 | elapsed:    1.1s finished
/data/leuven/352/vsc35289/miniconda3/envs/bttda-cuda/lib/python3.11/site-packages/pandas/core/internals/concat.py:314: RuntimeWarning: coroutine 'WSHandler.send_message' was never awaited
  unit = JoinUnit(nb)


In [5]:
results.to_csv('results/moabb_erp.csv')
results

,score,time,samples,subject,session,channels,n_sessions,dataset,pipeline
0,0.903377,145.371933,768.0,1,0,16,1,BrainInvaders2012,HODA
1,0.942526,217.494568,768.0,1,0,16,1,BrainInvaders2012,PARAFACDA
2,0.940565,237.892670,768.0,1,0,16,1,BrainInvaders2012,BTTDA
3,0.930973,93.988861,764.0,2,0,16,1,BrainInvaders2012,HODA
4,0.946500,229.975937,764.0,2,0,16,1,BrainInvaders2012,PARAFACDA
...,...,...,...,...,...,...,...,...,...
2518,0.967467,763.506897,4140.0,54,0,62,2,Lee2019-ERP,HODA
2519,0.951128,252.888809,4140.0,54,1,62,2,Lee2019-ERP,HODA
2520,0.978066,328.157410,4140.0,54,0,62,2,Lee2019-ERP,PARAFACDA
2521,0.970609,698.454041,4140.0,54,1,62,2,Lee2019-ERP,PARAFACDA


In [6]:
results = pd.read_csv('results/moabb_erp.csv')

In [7]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean')

dataset             pipeline 
BNCI2014-008        BTTDA        0.860964
                    HODA         0.851951
                    PARAFACDA    0.859321
BNCI2014-009        BTTDA        0.942909
                    HODA         0.932335
                    PARAFACDA    0.940898
BNCI2015-003        BTTDA        0.849914
                    HODA         0.827415
                    PARAFACDA    0.846299
BrainInvaders2012   BTTDA        0.909746
                    HODA         0.867054
                    PARAFACDA    0.907316
BrainInvaders2013a  BTTDA        0.920214
                    HODA         0.897819
                    PARAFACDA    0.916608
BrainInvaders2014a  BTTDA        0.878102
                    HODA         0.842333
                    PARAFACDA    0.873958
BrainInvaders2014b  BTTDA        0.906880
                    HODA         0.872029
                    PARAFACDA    0.896913
BrainInvaders2015a  BTTDA        0.935034
                    HODA         0.909677
    

In [8]:
results.groupby(['dataset', 'pipeline'])['score'].aggregate('mean').reset_index().groupby('pipeline')['score'].aggregate('mean')

pipeline
BTTDA        0.875208
HODA         0.855759
PARAFACDA    0.870106
Name: score, dtype: float64

In [9]:
df_diff = results.pivot(index=['subject', 'session', 'channels', 'n_sessions', 'samples', 'dataset'], columns='pipeline', values='score')
df_diff = df_diff.reset_index()
df_diff['score_diff'] = df_diff['BTTDA'] - df_diff['HODA']
df_diff

pipeline,subject,session,channels,n_sessions,samples,dataset,BTTDA,HODA,PARAFACDA,score_diff
0,1,0,8,1,4200.0,BNCI2014-008,0.824455,0.812371,0.824737,0.012084
1,1,0,8,1,5400.0,BNCI2015-003,0.796502,0.758171,0.790889,0.038330
2,1,0,16,1,480.0,BrainInvaders2013a,0.985156,0.933594,0.983750,0.051562
3,1,0,16,1,768.0,BrainInvaders2012,0.940565,0.903377,0.942526,0.037188
4,1,0,16,1,1188.0,BrainInvaders2014a,0.957428,0.884317,0.952183,0.073111
...,...,...,...,...,...,...,...,...,...,...
851,60,0,16,1,792.0,BrainInvaders2014a,0.973668,0.967554,0.974251,0.006115
852,61,0,16,1,1368.0,BrainInvaders2014a,0.762736,0.693179,0.782735,0.069557
853,62,0,16,1,1367.0,BrainInvaders2014a,0.534735,0.499819,0.560169,0.034916
854,63,0,16,1,420.0,BrainInvaders2014a,0.654286,0.634490,0.606735,0.019796


In [10]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'iframe'

def compare_score_plot(df, pipe1, pipe2):
    fig = px.scatter(df, x=pipe1, y=pipe2, color='dataset', facet_col='dataset', facet_col_wrap=5)
    fig.update_yaxes(scaleanchor="x")
    fig.update_xaxes(range=[.5, 1])
    fig.update_yaxes(range=[.5, 1])
    fig.add_shape(
        type="line",
        x0=0.5, y0=0.5, x1=1, y1=1,
        line=dict(color="gray", dash='dash'),
        layer="below" ,
        row='all', col='all', exclude_empty_subplots=True
    )
    

    return fig

fig = compare_score_plot(df_diff, 'HODA', 'BTTDA')
fig.update_layout(
    autosize=False,
    width=1800,
    height=1800,
)
fig.update_layout(showlegend=False)
fig